# All four cycles: what the ECMWF IFS store actually holds

The published GIK store
([`ecmwf_ifs_ens_aws_s3_icechunk_vd`](https://source.coop/e4drr-project/forecasts/ecmwf_ifs_ens_aws_s3_icechunk_vd))
indexes **all four daily forecast cycles** -- 00z, 06z, 12z and 18z -- across all
three schema eras. That is **5,230 date-cycles** and **1.61 PB** of referenced GRIB.

> **This notebook corrects an earlier one.** The previous version claimed the store
> held 00z only and was missing ~1.1 PB. That was wrong. It opened `{era}/00z`
> because an older notebook did, and read the `00z` in that path as a constraint
> rather than as one of four choices. A single `list(group.group_keys())` would have
> settled it. The measured archive sizes in that notebook were sound; the framing
> was not -- that 1.1 PB is **already indexed**, not absent.

What follows is what the store contains, verified group by group, plus the two real
gaps that the group listing does *not* reveal.


## 1. Enumerate the groups -- do not assume them

The store nests `{era}/{cycle}`. Ask it what it has:


In [ ]:
import asyncio
import threading
from concurrent.futures import ThreadPoolExecutor

import boto3
import numpy as np
import pandas as pd
import icechunk
import zarr
import xarray as xr
import gribberish.zarr  # noqa: F401 -- registers the Zarr v3 codec
from botocore import UNSIGNED
from botocore.config import Config

ENDPOINT = "https://data.source.coop"
BUCKET   = "e4drr-project"
PREFIX   = "forecasts/ecmwf_ifs_ens_aws_s3_icechunk_vd"
ARCHIVE  = "ecmwf-forecasts"

storage = icechunk.s3_storage(
    bucket=BUCKET, prefix=PREFIX, endpoint_url=ENDPOINT, region="us-east-1",
    anonymous=True, from_env=False, force_path_style=True,
)
cfg = icechunk.RepositoryConfig.default()
cfg.manifest = icechunk.ManifestConfig(
    preload=icechunk.ManifestPreloadConfig(max_total_refs=0, max_arrays_to_scan=0))
repo = icechunk.Repository.open(storage, config=cfg)
sess = repo.readonly_session("main")

root = zarr.open_group(sess.store, mode="r")
ERAS = list(root.group_keys())
for era in ERAS:
    cycles = sorted(zarr.open_group(sess.store, mode="r", path=era).group_keys())
    print(f"{era:5s} cycles: {cycles}")


0p4   cycles: ['00z', '06z', '12z', '18z']
49r1  cycles: ['00z', '06z', '12z', '18z']
50r1  cycles: ['00z', '06z', '12z', '18z']


Twelve groups, not three. The store has been four-cycle for some time -- the HEAD
commit below is a `50r1/18z` write.


In [ ]:
anc = list(repo.ancestry(branch="main"))
print(f"{len(anc):,} snapshots   HEAD {str(anc[0].id)[:16]}")
for c in anc[:3]:
    print(f"   {str(c.written_at)[:19]}  {c.message[:60]}")


3,787 snapshots   HEAD CSG8BX0ZMZMF8Q09
   2026-08-25 12:41:53  50r1/18z 20260823: 51 members, 410525 refs (time index 103)
   2026-08-25 12:40:31  50r1/18z 20260822: 51 members, 410525 refs (time index 102)
   2026-08-25 12:39:10  50r1/18z 20260821: 51 members, 410525 refs (time index 101)


## 2. The full inventory


In [ ]:
CYCLES = ["00z", "06z", "12z", "18z"]

def open_group(era, cycle):
    return xr.open_zarr(sess.store, group=f"{era}/{cycle}", zarr_format=3,
                        consolidated=False, chunks=None, decode_timedelta=True)

rows = []
for era in ["0p4", "49r1", "50r1"]:
    for cyc in CYCLES:
        d = open_group(era, cyc)
        t = pd.to_datetime(d.time.values)
        rows.append(dict(era=era, cycle=cyc, dates=len(t), steps=d.sizes["step"],
                         vars=len(d.data_vars), members=d.sizes["number"],
                         grid=f'{d.sizes["latitude"]}x{d.sizes["longitude"]}',
                         first=str(t.min().date()), last=str(t.max().date())))

inv = pd.DataFrame(rows)
print(inv.to_string(index=False))
print(f"\ntotal date-cycles indexed: {inv.dates.sum():,}")


  era cycle  dates  steps  vars  members      grid      first       last
  0p4   00z    401     85    19       51   451x900 2023-01-18 2024-02-28
  0p4   06z    400     49    19       51   451x900 2023-01-18 2024-02-27
  0p4   12z    399     85    19       51   451x900 2023-01-18 2024-02-27
  0p4   18z    399     49    19       51   451x900 2023-01-18 2024-02-27
 49r1   00z    804     85    59       51  721x1440 2024-02-29 2026-05-12
 49r1   06z    804     49    59       51  721x1440 2024-02-28 2026-05-11
 49r1   12z    804     85    59       51  721x1440 2024-02-28 2026-05-11
 49r1   18z    804     49    57       51  721x1440 2024-02-28 2026-05-11
 50r1   00z    103     85    54       51  721x1440 2026-05-13 2026-08-23
 50r1   06z    104     49    54       51  721x1440 2026-05-12 2026-08-23
 50r1   12z    104     85    54       51  721x1440 2026-05-12 2026-08-23
 50r1   18z    104     49    54       51  721x1440 2026-05-12 2026-08-23

total date-cycles indexed: 5,230


Three things in that table look like defects and only one is:

1. `steps` is **85 at 00z/12z, 49 at 06z/18z** -- correct, and explained in section 3.
2. `49r1/18z` has **57 vars** where its siblings have 59 -- section 4. It is the
   *honest* case.
3. The 06z/12z/18z groups start **one day earlier** than 00z, and `0p4` cycle date
   counts differ -- section 5. Also correct, and for a reason worth knowing.


## 3. Why the four cycles are not four equal quarters

ECMWF takes the ensemble to **+360 h at 00z and 12z**, but only to **+144 h at 06z
and 18z**. Same 51 members, same grid, roughly half the lead times:


In [ ]:
import re

s3 = boto3.client("s3", region_name="eu-central-1",
                  config=Config(signature_version=UNSIGNED))

def res_for(era):
    """Archive path component. The beta era is laid out differently."""
    return "0p4-beta" if era == "0p4" else "ifs/0p25"

def cycle_shape(date, cycle, era):
    n = sz = 0
    steps = set()
    for page in s3.get_paginator("list_objects_v2").paginate(
            Bucket=ARCHIVE, Prefix=f"{date}/{cycle}/{res_for(era)}/enfo/"):
        for o in page.get("Contents", []):
            if not o["Key"].endswith(".grib2"):
                continue
            n += 1
            sz += o["Size"]
            m = re.search(r"-(\d+)h-enfo", o["Key"])
            if m:
                steps.add(int(m.group(1)))
    return n, sz, len(steps), (max(steps) if steps else 0)

base = None
for cyc in CYCLES:
    n, sz, nstep, maxstep = cycle_shape("20260820", cyc, "50r1")
    base = base or sz
    print(f"  {cyc}  {n:3d} files  {sz/1e9:6.1f} GB  {nstep:2d} steps to +{maxstep}h"
          f"   ratio to 00z {sz/base:.3f}")


  00z   87 files   563.0 GB  85 steps to +360h   ratio to 00z 1.000
  06z   49 files   323.3 GB  49 steps to +144h   ratio to 00z 0.574
  12z   87 files   562.4 GB  85 steps to +360h   ratio to 00z 0.999
  18z   49 files   323.1 GB  49 steps to +144h   ratio to 00z 0.574


    1.000 + 0.574 + 0.999 + 0.574 = 3.15

So a day of forecasts is **3.15x** a 00z run, not 4x. Anyone sizing four-cycle
coverage by multiplying 00z by four overshoots by ~27%. The 85/49/85/49 ladder is
identical in every era, `0p4-beta` included.


## 4. The real defect: an array that exists but was never written

`49r1/18z` exposes 57 variables against 59 elsewhere. The two absentees are `mn2t6`
and `mx2t6` -- 6-hourly minimum and maximum 2 m temperature.

A missing variable is easy to handle: `.data_vars` says so, and code that needs it
fails loudly. The problem is what the sibling cycle does instead.


In [ ]:
async def written_dates(path):
    """Time indices this array actually has chunks for -- manifest only, no data read."""
    out = set()
    async for c in sess.chunk_coordinates(path):
        out.add(tuple(c)[0])
    return out

for cyc in CYCLES:
    d = open_group("49r1", cyc)
    for v in ("mn2t6", "mx2t6"):
        if v not in d.data_vars:
            print(f"  49r1/{cyc}/{v:6s}: ARRAY ABSENT        -- honest: .data_vars says so")
            continue
        n = len(asyncio.run(written_dates(f"/49r1/{cyc}/{v}")))
        flag = "  <-- EMPTY, every read returns NaN" if n == 0 else ""
        print(f"  49r1/{cyc}/{v:6s}: {n:4d} dates written{flag}")


  49r1/00z/mn2t6 :  546 dates written
  49r1/00z/mx2t6 :  546 dates written
  49r1/06z/mn2t6 :    0 dates written  <-- EMPTY, every read returns NaN
  49r1/06z/mx2t6 :    0 dates written  <-- EMPTY, every read returns NaN
  49r1/12z/mn2t6 :  546 dates written
  49r1/12z/mx2t6 :  546 dates written
  49r1/18z/mn2t6 : ARRAY ABSENT        -- honest: .data_vars says so
  49r1/18z/mx2t6 : ARRAY ABSENT        -- honest: .data_vars says so


In [ ]:
d = open_group("49r1", "06z")
t = pd.to_datetime(d.time.values)
ti = int(np.argmin(np.abs(t - pd.Timestamp("2025-06-01"))))
field = d["mn2t6"].isel(time=ti, number=0, step=2).values

print(f"49r1/06z mn2t6 @ {t[ti].date()}")
print(f"  shape {field.shape}   finite fraction {np.isfinite(field).mean():.3f}")
print("  -> the selection SUCCEEDS and returns an all-NaN field.")
print("     Nothing in the API distinguishes 'never written' from 'no data'.")


49r1/06z mn2t6 @ 2025-06-01
  shape (721, 1440)   finite fraction 0.000
  -> the selection SUCCEEDS and returns an all-NaN field.
     Nothing in the API distinguishes 'never written' from 'no data'.


### Why this matters more than a missing variable

The `time` axis is preallocated to the whole era, so an unwritten date is not absent:
it is on the axis, `.sel()` finds it, and it reads back as NaN. For `49r1/06z/mn2t6`
that is true of **every** date -- the array was created and never filled.

A consumer computing, say, a heatwave index over 06z would get NaN everywhere and no
signal that the variable was never published for that cycle. `49r1/18z`, which simply
omits the array, is the safer of the two behaviours.

**Check coverage from the manifest, not from `.data_vars`.** `chunk_coordinates` is
cheap -- it reads the manifest, never the chunks -- and it is the only way to tell an
empty array from a full one without moving terabytes:


In [ ]:
d = open_group("49r1", "00z")
t = pd.to_datetime(d.time.values)
n = len(asyncio.run(written_dates("/49r1/00z/mn2t6")))
idx = sorted(asyncio.run(written_dates("/49r1/00z/mn2t6")))
print(f"49r1/00z/mn2t6 covers {n} of {len(t)} dates")
print(f"  first written {t[idx[0]].date()}   last {t[idx[-1]].date()}")
print(f"  the {len(t)-n} earlier dates on the axis read as NaN")


49r1/00z/mn2t6 covers 546 of 804 dates
  first written 2024-11-13   last 2026-05-12
  the 258 earlier dates on the axis read as NaN


So even where the variable *is* populated it starts partway through the era
(2024-11-13), because ECMWF began publishing it then. Partial coverage is the norm
for the newer fields, and it is invisible without a manifest check.


## 5. The date offsets are upstream, not build errors

Two patterns looked like off-by-one bugs. Neither is.


In [ ]:
sets = {}
for cyc in CYCLES:
    d = open_group("0p4", cyc)
    sets[cyc] = set(pd.to_datetime(d.time.values).date)

ref = sets["00z"]
for cyc in ("06z", "12z", "18z"):
    missing = sorted(ref - sets[cyc])
    extra = sorted(sets[cyc] - ref)
    print(f"0p4/{cyc}: {len(sets[cyc])} dates | absent vs 00z: {[str(x) for x in missing]}"
          f" | present but not in 00z: {[str(x) for x in extra]}")


0p4/06z: 400 dates | absent vs 00z: ['2024-02-28'] | present but not in 00z: []
0p4/12z: 399 dates | absent vs 00z: ['2023-04-26', '2024-02-28'] | present but not in 00z: []
0p4/18z: 399 dates | absent vs 00z: ['2023-04-26', '2024-02-28'] | present but not in 00z: []


In [ ]:
for date in ("20230425", "20230426", "20240228"):
    r = s3.list_objects_v2(Bucket=ARCHIVE, Prefix=f"{date}/", Delimiter="/")
    counts = {}
    for p in r.get("CommonPrefixes", []):
        cyc = p["Prefix"].split("/")[-2]
        n = 0
        for page in s3.get_paginator("list_objects_v2").paginate(
                Bucket=ARCHIVE, Prefix=f"{date}/{cyc}/0p4-beta/enfo/"):
            n += sum(1 for o in page.get("Contents", []) if o["Key"].endswith(".grib2"))
        counts[cyc] = n
    print(f"{date}  0p4-beta files per cycle: {counts}")


20230425  0p4-beta files per cycle: {'00z': 87, '06z': 49, '12z': 87, '18z': 49}
20230426  0p4-beta files per cycle: {'00z': 87, '06z': 49}
20240228  0p4-beta files per cycle: {'00z': 87, '06z': 0, '12z': 0, '18z': 0}


**2023-04-26** -- ECMWF published 00z and 06z only. No 12z or 18z exists to index.
The store reflects the archive faithfully.

**2024-02-28 -- the era boundary falls mid-day.** The 00z run that morning was the
last `0p4-beta` output; by 06z the same day ECMWF had switched to the 0.25 deg
`49r1` stream. That is why the `0p4` cycles end a day apart, and why in the
inventory table `49r1/06z`, `/12z` and `/18z` all begin on **2024-02-28** while
`49r1/00z` begins on **2024-02-29**:


In [ ]:
for era in ("0p4", "49r1"):
    row = {}
    for cyc in CYCLES:
        t = pd.to_datetime(open_group(era, cyc).time.values)
        row[cyc] = f"{t.min().date()}..{t.max().date()}"
    print(f"{era:5s} " + "   ".join(f"{c} {v}" for c, v in row.items()))
print("\n-> 2024-02-28 00z is 0p4; 2024-02-28 06z/12z/18z are already 49r1.")
print("   An era is not a whole number of days.")


0p4   00z 2023-01-18..2024-02-28   06z 2023-01-18..2024-02-27   12z 2023-01-18..2024-02-27   18z 2023-01-18..2024-02-27
49r1  00z 2024-02-29..2026-05-12   06z 2024-02-28..2026-05-11   12z 2024-02-28..2026-05-11   18z 2024-02-28..2026-05-11

-> 2024-02-28 00z is 0p4; 2024-02-28 06z/12z/18z are already 49r1.
   An era is not a whole number of days.


## 6. How much data this is

Measured by listing every `.grib2` object for every date-cycle in the store --
5,232 listings, 0 failures. Not extrapolated from a sample.


In [ ]:
dates_of = {(e, c): [pd.Timestamp(t).strftime("%Y%m%d")
                     for t in open_group(e, c).time.values]
            for e in ["0p4", "49r1", "50r1"] for c in CYCLES}

_local = threading.local()
def client():
    """One boto3 client per thread -- they are not safe to share."""
    if not hasattr(_local, "c"):
        _local.c = boto3.client("s3", region_name="eu-central-1",
                                config=Config(signature_version=UNSIGNED,
                                              max_pool_connections=8,
                                              retries={"max_attempts": 5}))
    return _local.c

def measure(job):
    date, cycle, era = job
    n = sz = 0
    try:
        for page in client().get_paginator("list_objects_v2").paginate(
                Bucket=ARCHIVE, Prefix=f"{date}/{cycle}/{res_for(era)}/enfo/"):
            for o in page.get("Contents", []):
                if o["Key"].endswith(".grib2"):
                    n += 1
                    sz += o["Size"]
    except Exception:
        return date, cycle, era, -1, 0      # -1 marks a failure, never a silent 0
    return date, cycle, era, n, sz

jobs = [(d, c, e) for (e, c), ds in dates_of.items() for d in ds]
print(f"{len(jobs):,} listings ...")
with ThreadPoolExecutor(max_workers=24) as ex:
    rows = list(ex.map(measure, jobs))

df = pd.DataFrame(rows, columns=["date", "cycle", "era", "files", "bytes"])
failed = int((df.files < 0).sum())
print(f"done. failed listings: {failed}")
assert failed == 0, "a failed listing counts 0 bytes and would silently deflate the total"


5,230 listings ...
done. failed listings: 0


That assertion is the point. A failed listing returns zero bytes, which is
indistinguishable from an empty cycle -- it would quietly shrink the total instead of
raising. Count failures separately and assert on them.


In [ ]:
tb = df.pivot_table(index="era", columns="cycle", values="bytes", aggfunc="sum") / 1e12
tb = tb.reindex(index=["0p4", "49r1", "50r1"], columns=CYCLES)
tb["TOTAL"] = tb.sum(axis=1)
tb.loc["ALL"] = tb.sum()
print("referenced GRIB, TB\n")
print(tb.round(1).to_string())
print(f"\ngrand total: {df.bytes.sum()/1e15:.3f} PB across {len(df):,} date-cycles")


referenced GRIB, TB

cycle    00z    06z    12z    18z   TOTAL
era
0p4     64.3   36.8   63.8   36.6   201.5
49r1   389.7  224.8  390.0  224.9  1229.4
50r1    58.5   33.7   58.5   33.7   184.4
ALL    512.5  295.3  512.3  295.2  1615.3

grand total: 1.615 PB across 5,230 date-cycles


00z and 12z are twins (512.5 vs 512.3 TB); 06z and 18z are twins (295.3 vs 295.2).
`49r1` is 76% of the whole, being the longest era at the finer grid.


## 7. Three numbers people conflate

| measure | value | what it means |
|---|---|---|
| store objects on source.coop | ~15 GB | what is hosted -- pointers, not data |
| **referenced GRIB, all four cycles** | **1.62 PB** | bytes pulled reading the store once |
| dense float32 (`ds.nbytes`) | ~9 PB | every cell materialized -- **misleading** |

`ds.nbytes` assumes a dense array. GRIB is packed and the archive is not dense, so
it overstates by roughly 5-6x. Quote the referenced figure.


In [ ]:
dense = sum(open_group(e, c).nbytes
            for e in ["0p4", "49r1", "50r1"] for c in CYCLES)
ref = df.bytes.sum()
print(f"dense float32, all 12 groups : {dense/1e15:6.2f} PB")
print(f"referenced GRIB (packed)     : {ref/1e15:6.3f} PB")
print(f"inflation factor             : {dense/ref:6.1f}x")


dense float32, all 12 groups :   8.98 PB
referenced GRIB (packed)     :  1.615 PB
inflation factor             :    5.6x


## 8. Per-date cost, and why an era mean misleads

Sizing a future build needs GB per date. That number is not stable within an era --
it steps when ECMWF changes what it publishes:


In [ ]:
z = df[df.cycle == "00z"].copy()
z["month"] = pd.to_datetime(z.date).dt.to_period("M")

per_era = z.groupby("era")["bytes"].agg(["mean", "min", "max"]) / 1e9
per_era["dates"] = z.groupby("era").size()
print("00z GB per date\n")
print(per_era.round(1).to_string())

print("\nmonthly mean, showing the steps (GB):")
m = z.groupby("month")["bytes"].mean() / 1e9
print(m.loc[["2023-06", "2023-08", "2024-02", "2024-12", "2025-04", "2026-08"]].round(0).to_string())


00z GB per date

       mean    min    max  dates
era
0p4   160.3  116.0  228.4    401
49r1  484.7  382.4  557.1    804
50r1  567.6  561.3  575.4    103

monthly mean, showing the steps (GB):
month
2023-06    228
2023-08    116
2024-02    118
2024-12    422
2025-04    553
2026-08    567


Two discontinuities, both *inside* an era rather than at a boundary:

- **~Jul 2023**, `0p4`: 228 -> 116 GB/date, a halving.
- **late 2024 / early 2025**, `49r1`: ~420 -> ~553 GB/date -- the same window in which
  `mn2t6`/`mx2t6` first appear (2024-11-13, section 4).

So `0p4`'s 160 GB/date mean describes no real date; it averages two regimes differing
by 2x. For planning, use the recent figure for the era you will build in --
`50r1` at **568 GB/date**, tight at 561-575 -- not the lifetime mean.


## 9. Gotchas

- **Enumerate `group_keys()`; do not assume the layout.** The path is `{era}/{cycle}`
  and there are twelve groups. Hardcoding `00z` is how the previous version of this
  notebook concluded three quarters of the store did not exist.
- **An array can exist and be empty.** `49r1/06z/mn2t6` selects fine and returns
  all-NaN. `.data_vars` cannot tell you; `session.chunk_coordinates` can, from the
  manifest, without reading a chunk.
- **Partial coverage is normal.** Even where populated, `mn2t6` starts 2024-11-13 --
  258 dates of the era read as NaN.
- **`06z`/`18z` stop at +144 h.** Treating the cycles as equal overstates by ~27%.
- **An era is not a whole number of days.** The 0p4 -> 49r1 switch happens between
  the 00z and 06z runs of 2024-02-28.
- **The archive path differs by era**: `0p4-beta`, then `ifs/0p25`. A wrong path
  returns an empty listing, not an error.
- **A failed listing looks like an empty cycle.** Both yield zero bytes.
- **Do not quote `ds.nbytes`** -- 5.6x the packed size, describing an array that does
  not exist.
- **Import `gribberish.zarr`** before any chunk read, or opening a group fails with
  `UnknownCodecError: 'gribberish'`.
- **Pass `decode_timedelta=True`** to `open_zarr`, or xarray warns about `step`.

---

### On the earlier size figure

An older notebook quoted ~620 TB referenced for 00z over 1,246 dates. This sweep
measures **512 TB** for 00z over 1,308 dates -- fewer bytes across more dates. The
two were derived differently and I have not reconciled them. The figure here is a
direct sum of every `.grib2` object under the `enfo` prefix for exactly the dates the
store carries, zero failed listings, and its implied per-date means (`50r1` 568 GB)
match spot checks of individual dates.
